In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
from pathlib import Path
import urllib.request


data_path = Path("names.txt")
if not data_path.exists():
    url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
    urllib.request.urlretrieve(url, data_path)

words = data_path.read_text().splitlines()
print(f"Loaded {len(words)} names from {data_path}")

Loaded 32033 names from names.txt


In [3]:
alphabets=sorted(list(set(''.join(words))))
stringToIndex={s:i+1 for i,s in enumerate(alphabets)}
IndexToString={s+1:i for s,i in enumerate(alphabets)}
stringToIndex["."]=0
IndexToString[0]="."
stringToIndex

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [ ]:
# build the dataset
itos=IndexToString
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stringToIndex[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

In [12]:
print(X,Y)

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]]) tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])


In [14]:
#lookup table 
C=torch.randn((27,2))
print(C)

tensor([[ 1.2856,  0.1538],
        [ 0.7423,  0.9530],
        [ 0.1225, -0.1172],
        [-0.6522,  1.0197],
        [ 1.1482,  0.3266],
        [-0.6900, -1.3065],
        [ 0.9572, -0.9978],
        [-0.8776,  1.0532],
        [-1.4415,  0.6404],
        [ 0.4866,  1.9576],
        [ 0.6459, -1.2804],
        [ 1.1129, -0.6472],
        [-0.3831, -1.4623],
        [ 0.3498, -1.1539],
        [-1.8122,  0.3275],
        [-1.2127,  0.0837],
        [-1.4552,  0.9641],
        [ 0.2686, -1.6312],
        [ 1.3246,  0.6346],
        [-0.8383,  0.4047],
        [-0.1774, -0.1662],
        [-0.8455,  1.5892],
        [-0.3036, -2.0385],
        [-0.3812,  0.7415],
        [ 0.6186, -0.0246],
        [-1.8420,  1.0484],
        [-0.2883,  0.0726]])


In [6]:
(F.one_hot(torch.tensor(5),num_classes=27).float())@C

tensor([0.2471, 0.5857])

In [7]:
C[5]

tensor([0.2471, 0.5857])

In [8]:
embeddings=C[X]